In [27]:
#!pip install statsmodels

# \-\-\-\-\-\-\-\-\-\- Importar librerías \-\-\-\-\-\-\-\-\-\-

In [28]:
import os
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import plotly.graph_objects as go
import shutil

## \-\-\-\-\-\-\-\-\-\- 1\) Cargar y limpiar los datos\-\-\-\-\-\-\-\-\-\-

## Cargar datos

In [29]:
path = "../data/processed/kakebo_merged.csv"

# Condicional 'if' para leer archivo .csv

if path == "../data/processed/kakebo_merged.csv":
    df = pd.read_csv(path, sep=";", encoding="utf-8")
    print(f"Nota: El archivo '{path}' fue cargado exitosamente.")
else:
    raise ValueError(f"Error: El archivo '{path}' no fue cargado exitosamente." )

Nota: El archivo '../data/processed/kakebo_merged.csv' fue cargado exitosamente.


## Limpiar y parsear datos

In [30]:
df.columns = [c.strip().upper() for c in df.columns]

def parse_monto(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.replace(".", "")      # miles
    s = s.replace(",", ".")     # decimal
    return float(s)

df["MONTO"] = df["MONTO"].apply(parse_monto)

df["MES_NOMBRE"] = df["MES"].astype(str).str.replace("MES", "", regex=False).str.strip().str.upper()

mes_map = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12
}
df["MES_NUM"] = df["MES_NOMBRE"].map(mes_map)
df["FECHA"] = pd.to_datetime(dict(year=df["AÑO"], month=df["MES_NUM"], day=1))
df = df.sort_values("FECHA")

# Entrenar SOLO con datos reales
if "TIPO_DATO" in df.columns:
    train_df = df[df["TIPO_DATO"].astype(str).str.upper() == "REAL"].copy()
    if train_df.empty:
        train_df = df.copy()
else:
    train_df = df.copy()
y = train_df.set_index("FECHA")["MONTO"].asfreq("MS")

## \-\-\-\-\-\-\-\-\-\- 2\) Implementar y entrenar modelo ARIMA \-\-\-\-\-\-\-\-\-\-

In [31]:
order = (1, 1, 1)
model = ARIMA(y, order=order, enforce_stationarity=False, enforce_invertibility=False)
res = model.fit()

## \-\-\-\-\-\-\-\-\-\- 3\) Pronóstico a 12 meses \-\-\-\-\-\-\-\-\-\-

In [32]:
steps = 12
forecast_res = res.get_forecast(steps=steps)

yhat = forecast_res.predicted_mean #uso de la media para hacer las predicciones
ci = forecast_res.conf_int(alpha=0.05)  # uso del 95% en el rango Intervalo de Confianza

## \-\-\-\-\-\-\-\-\-\- 4\) Predicción con intervalo de confianza IC\-\-\-\-\-\-\-\-\-\-

In [33]:
# Histórico
hist_out = (
    y.reset_index()
     .rename(columns={"FECHA": "fecha", "MONTO": "monto"})
)
hist_out["tipo_dato"] = "REAL"
hist_out["lower_95"] = np.nan
hist_out["upper_95"] = np.nan

# Pronóstico
pred_out = pd.DataFrame({
    "fecha": yhat.index,
    "monto": yhat.values,
    "tipo_dato": "PREDICCION",
    "lower_95": ci.iloc[:, 0].values,
    "upper_95": ci.iloc[:, 1].values
})

out = pd.concat([hist_out, pred_out], ignore_index=True).sort_values("fecha")

# Opcional: formatear fecha YYYY-MM-DD
out["fecha"] = out["fecha"].dt.strftime("%Y-%m-%d")

### \-\-\-\-\-\-\-\-\-\- 4\.1\) Exportar histórico \+ predicción \-\-\-\-\-\-\-\-\-\-

In [34]:
out_path = "../data/processed/kakebo_pred_hist.csv"
out.to_csv(out_path, index=False, encoding="utf-8")

print(f"Archivo exportado: {out_path}")
print(out.tail(15).to_string(index=False))

Archivo exportado: ../data/processed/kakebo_pred_hist.csv
     fecha        monto  tipo_dato     lower_95     upper_95
2025-12-01 6.068471e+06       REAL          NaN          NaN
2026-01-01 8.807436e+06       REAL          NaN          NaN
2026-02-01 5.569606e+06       REAL          NaN          NaN
2026-03-01 7.505838e+06 PREDICCION 4.803120e+06 1.020856e+07
2026-04-01 6.384554e+06 PREDICCION 3.492295e+06 9.276813e+06
2026-05-01 7.033897e+06 PREDICCION 3.518282e+06 1.054951e+07
2026-06-01 6.657858e+06 PREDICCION 2.859686e+06 1.045603e+07
2026-07-01 6.875624e+06 PREDICCION 2.688457e+06 1.106279e+07
2026-08-01 6.749515e+06 PREDICCION 2.276170e+06 1.122286e+07
2026-09-01 6.822546e+06 PREDICCION 2.042986e+06 1.160211e+07
2026-10-01 6.780253e+06 PREDICCION 1.733558e+06 1.182695e+07
2026-11-01 6.804745e+06 PREDICCION 1.493078e+06 1.211641e+07
2026-12-01 6.790561e+06 PREDICCION 1.232790e+06 1.234833e+07
2027-01-01 6.798775e+06 PREDICCION 1.001874e+06 1.259568e+07
2027-02-01 6.794019e+06 PRE

## \-\-\-\-\-\-\-\-\-\- Gráficas de serie temporal \-\-\-\-\-\-\-\-\-\-

In [35]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pd.to_datetime(hist_out["fecha"]),
    y=hist_out["monto"].values,
    mode="lines+markers",
    name="Datos Históricos (REALES)"
))

fig.add_trace(go.Scatter(
    x=yhat.index, y=yhat.values,
    mode="lines+markers",
    name="Predicción"
))

fig.add_trace(go.Scatter(
    x=ci.index,
    y=ci.iloc[:, 0].values,
    mode="lines",
    line=dict(width=0),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=ci.index,
    y=ci.iloc[:, 1].values,
    mode="lines",
    fill="tonexty",
    line=dict(width=0),
    name="Rango de Predicción - (IC 95%)"
))

fig.update_layout(
    title=f"Predicciones de gastos KAKEBO 2026-2027",
    title_x=0.5,
    title_xanchor="center",
    xaxis_title="Fecha",
    yaxis_title="Monto",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

Se habla de que el Intervalo de confianza, es el rango que estima dónde podría estar ese valor verdadero dentro de la franja morada, el punto rojo, es el valor de la predicción estimada a gastar durante los siguientes meses\.

# \-\-\-\-\-\-\-\-\-\- Exportar para el tablero de PowerBI \-\-\-\-\-\-\-\-\-\-

In [36]:
df = pd.read_csv("../data/processed/kakebo_pred_hist.csv")

df = df.drop(columns=["lower_95", "upper_95"], errors="ignore").rename(columns={
    "fecha": "FECHA",
    "monto": "MONTO",
    "tipo_dato": "TIPO_DATO",
})

df["MONTO"] = pd.to_numeric(df["MONTO"], errors="coerce").round(0).astype("Int64")

# sin la palabra COP
def formato_pesos_colombianos(x):
    if pd.isna(x):
        return None
    return f"$ {int(x):,}".replace(",", ".")

df["MONTO_COP"] = df["MONTO"].apply(formato_pesos_colombianos)



df.to_csv("../data/processed/kakebo_pred_pbix.csv", index=False)
print(df[["FECHA", "MONTO_COP", "TIPO_DATO"]].head())

        FECHA     MONTO_COP TIPO_DATO
0  2025-01-01  $ 14.057.968      REAL
1  2025-02-01   $ 3.310.580      REAL
2  2025-03-01   $ 3.072.536      REAL
3  2025-04-01   $ 3.455.498      REAL
4  2025-05-01   $ 3.385.952      REAL


# \-\-\-\-\-\-\-\-\-\- Exportar gráfico a formato HTML \-\-\-\-\-\-\-\-\-\-

In [37]:


fig.write_html("../dashboards/arima_forecast.html")
print("Gráfico exportado a: arima_forecast.html")


#exportar informe a la carpeta del dashboard de REDOHIS
output_path = r"C:\xampp\htdocs\REDOHIS\modules\dashboard\arima_forecast.html"
shutil.copy("../dashboards/arima_forecast.html", output_path)
print(f"Gráfico también exportado a: {output_path}")

Gráfico exportado a: arima_forecast.html
Gráfico también exportado a: C:\xampp\htdocs\REDOHIS\modules\dashboard\arima_forecast.html


Fuente: https://github\.com/copilot/share/0a1e4036\-40a0\-84f6\-b900\-260a20a209e2

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e741afa3-dba5-4ecd-ae0c-4881f669c8b0' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>